# Note:
We will conduct three pairwise comparisons for each of the 26 crime types: pre-COVID $\rightarrow$ COVID, COVID $\rightarrow$ post-COVID, and pre-COVID $\rightarrow$ post-COVID PCA, to provide a visual representation of each era, providing a temporal trajectory map. This will allow us to see if the internal structure and relationships between the 26 crime types "stretched," "shrunk," or "rotated" as society moved through the pandemic.

In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.feather as feather
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
from sklearn.decomposition import PCA
from clusteval import clusteval
from scatterd import scatterd
from importlib.metadata import version

# python source path
sys.path.append('../Src/')
# Set seed
SEED = 1776

# custom python
import plot
import utils
import zscore_anomaly

# Create a dictionary of versions
versions = {
    "Python": sys.version.split()[0],
    "Pandas": pd.__version__,
    "NumPy": np.__version__,
    "Pyarrow": pa.__version__, 
    "Seaborn": sns.__version__,
    "Matplot": sys.modules['matplotlib'].__version__,
    "Sk-Learn": sys.modules['sklearn'].__version__,
    "Clusteval": version("clusteval"),
    "Scatterd": version("scatterd")
}

# Display as a clean DataFrame
df_versions = pd.DataFrame(list(versions.items()), columns=['Library', 'Version'])
print(df_versions)

# display all columns
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
# Remove scientific notation
np.set_printoptions(suppress=True, precision=4, linewidth=100)
# reset options
# pd.reset_option('display.max_columns')

# Use a single Arrow string & int64 type instance to save memory
arrow_string = pd.ArrowDtype(pa.string())
arrow_int32 = pd.ArrowDtype(pa.int32())
arrow_cat8 = pd.ArrowDtype(pa.dictionary(index_type=pa.int8(), value_type=pa.string()))
arrow_cat16 = pd.ArrowDtype(pa.dictionary(index_type=pa.int16(), value_type=pa.string()))
arrow_cat32 = pd.ArrowDtype(pa.dictionary(index_type=pa.int32(), value_type=pa.string()))

     Library Version
0     Python  3.13.9
1     Pandas   2.3.3
2      NumPy   2.3.4
3    Pyarrow  22.0.0
4    Seaborn  0.13.2
5    Matplot  3.10.7
6   Sk-Learn   1.8.0
7  Clusteval   2.2.6
8   Scatterd   1.4.1


## Data Import

In [2]:
# PyArrow's version of the 'arrow' backend
df = feather.read_feather('../Data/crime_data_covid.feather', memory_map=True, types_mapper=pd.ArrowDtype)
# display
df.head()

,case_number,date,block,iucr,primary_type,description,location_description,arrest,domestic,beat,district,sector,ward,community_code,community_name,community_area,fbi_code,x_coordinate,y_coordinate,year,latitude,longitude,month,day_of_week,quarter,year_quarter,time_of_day,fbi_code_desc,fbi_index_code,district_location,year_week,era,Indexed
0,01G050460,2001-01-24 20:45:00,072XX S RIDGELAND AV,1811,NARCOTICS,POSS: CANNABIS 30GMS OR LESS,SIDEWALK,True,False,0324,003,1,<NA>,<NA>,<NA>,<NA>,18,1189075,1857566,2001,41.764219,-87.582549,January,Wednesday,Q1,2001-Q1,Night,Drug Abuse Violations,False,Grand Crossing,2001-04,pre_covid,N
1,03J493690,2003-07-12 17:00:00,075XX S DOBSON AVE,0890,THEFT,FROM BUILDING,APARTMENT,False,False,0624,006,2,08,69,GREATER GRAND CROSSING,98853167.7093,06,1184198,1855214,2003,41.75788,-87.600498,July,Saturday,Q3,2003-Q3,Evening,Larceny – Theft,True,Gresham,2003-28,pre_covid,I
2,04X245238,2004-12-13 21:15:00,006XX N RIDGEWAY AVE,2024,NARCOTICS,POSS: HEROIN(WHITE),SIDEWALK,True,False,1122,011,4,27,23,HUMBOLDT PARK,100480876.502,18,1151273,1903996,2004,41.892451,-87.719888,December,Monday,Q4,2004-Q4,Night,Drug Abuse Violations,False,Harrison,2004-51,pre_covid,N
3,07C115980,2006-03-31 09:15:00,026XX N NARRAGANSETT AVE,0610,BURGLARY,FORCIBLE ENTRY,APARTMENT,False,False,2512,025,5,29,19,BELMONT CRAGIN,109099414.689,05,1133296,1916864,2006,41.928096,-87.78561,March,Friday,Q1,2006-Q1,Morning,Burglary,True,Grand Central,2006-13,pre_covid,I
4,07HN36467,2007-05-25 14:51:00,022XX N LA CROSSE AVE,1812,NARCOTICS,POSS: CANNABIS MORE THAN 30GMS,RESIDENCE,True,False,2522,025,5,31,19,BELMONT CRAGIN,109099414.689,18,1143697,1914371,2007,41.921066,-87.747452,May,Friday,Q2,2007-Q2,Afternoon,Drug Abuse Violations,False,Grand Central,2007-21,pre_covid,N


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8469443 entries, 0 to 8469442
Data columns (total 33 columns):
 #   Column                Dtype                                                       
---  ------                -----                                                       
 0   case_number           string[pyarrow]                                             
 1   date                  timestamp[s][pyarrow]                                       
 2   block                 string[pyarrow]                                             
 3   iucr                  string[pyarrow]                                             
 4   primary_type          string[pyarrow]                                             
 5   description           string[pyarrow]                                             
 6   location_description  string[pyarrow]                                             
 7   arrest                bool[pyarrow]                                               
 8   do

## The Timeline Definition
* To ensure the analysis is accurate, we define the three eras based on global lockdown patterns:
    * Pre-COVID: January 2001 – February 2020
    * COVID Era: March 2020 – December 31, 2022
    * Post-COVID: January 2023 – Present

In [4]:
# Create the new column using .dt.strftime (Unit of Analysis: Year & Month)
df['year_month'] = df['date'].dt.strftime('%Y%m')

# convert year_month to int
df.year_month = df.year_month.astype(arrow_int32)
# check the datatype
print(df.year_month.dtype)
# display
df[['date','year_month']].head()

int32[pyarrow]


,date,year_month
0,2001-01-24 20:45:00,200101
1,2003-07-12 17:00:00,200307
2,2004-12-13 21:15:00,200412
3,2006-03-31 09:15:00,200603
4,2007-05-25 14:51:00,200705


## Data Prep
- Which crimes share the same time‑series shape across 2001–2025, including COVID and post‑COVID structural shifts?

In [5]:
# Time & Crime Type
master_matrix = df.groupby(['year_month', 'fbi_code_desc']).size()
master_matrix.sort_index(inplace=True)

# # Replace index 'year_month' to int32
# master_matrix.index = master_matrix.index.set_levels(
#     master_matrix.index.get_level_values('year_month').astype(arrow_int32), 
#     level='year_month'
# )
# Returns a Series where the index is the level name and the value is the dtype
print(master_matrix.index.dtypes)
# Multiindex name
print(master_matrix.index.names)
print(master_matrix.index.min(), master_matrix.index.max(), "\n")

# convert this Series to a DataFrame
master_matrix_flat = master_matrix.reset_index(name="crime_count")
master_matrix_flat.head()

year_month                                                     int32[pyarrow]
fbi_code_desc    dictionary<values=string, indices=int32, ordered=0>[pyarrow]
dtype: object
['year_month', 'fbi_code_desc']
(200101, 'Aggravated Assault') (202512, 'Weapons Violations') 



,year_month,fbi_code_desc,crime_count
0,200101,Aggravated Assault,546
1,200101,Aggravated Battery,967
2,200101,Arson,67
3,200101,Burglary,1934
4,200101,Criminal Sexual Assault,236


In [6]:
from scipy.stats import zscore

# Prevent Arrow Error
master_matrix_flat.fbi_code_desc = master_matrix_flat.fbi_code_desc.astype(str)

# Pivot
z_df_pivot = master_matrix_flat.pivot(index='fbi_code_desc', columns='year_month', values='crime_count').fillna(0)

# print shape
print("DataFrame shape:", z_df_pivot.shape, "\n")

DataFrame shape: (26, 300) 



In [7]:
# Z-score each Crime Type
z_row = zscore(z_df_pivot, axis=1)
# Convert into DataFrame
z_row_df = pd.DataFrame(z_row, index=z_df_pivot.index, columns=z_df_pivot.columns)
print("Colun Names:\n", z_row_df.columns, "\n")
# Display
z_row_df.head()

Colun Names:
 Index([200101, 200102, 200103, 200104, 200105, 200106, 200107, 200108, 200109,
       200110,
       ...
       202503, 202504, 202505, 202506, 202507, 202508, 202509, 202510, 202511,
       202512],
      dtype='int32[pyarrow]', name='year_month', length=300) 



year_month,200101,200102,200103,200104,200105,200106,200107,200108,200109,200110,200111,200112,200201,200202,200203,200204,200205,200206,200207,200208,200209,200210,200211,200212,200301,200302,200303,200304,200305,200306,200307,200308,200309,200310,200311,200312,200401,200402,200403,200404,200405,200406,200407,200408,200409,200410,200411,200412,200501,200502,200503,200504,200505,200506,200507,200508,200509,200510,200511,200512,200601,200602,200603,200604,200605,200606,200607,200608,200609,200610,200611,200612,200701,200702,200703,200704,200705,200706,200707,200708,200709,200710,200711,200712,200801,200802,200803,200804,200805,200806,200807,200808,200809,200810,200811,200812,200901,200902,200903,200904,200905,200906,200907,200908,200909,200910,200911,200912,201001,201002,201003,201004,201005,201006,201007,201008,201009,201010,201011,201012,201101,201102,201103,201104,201105,201106,201107,201108,201109,201110,201111,201112,201201,201202,201203,201204,201205,201206,201207,201208,201209,201210,201211,201212,201301,201302,201303,201304,201305,201306,201307,201308,201309,201310,201311,201312,201401,201402,201403,201404,201405,201406,201407,201408,201409,201410,201411,201412,201501,201502,201503,201504,201505,201506,201507,201508,201509,201510,201511,201512,201601,201602,201603,201604,201605,201606,201607,201608,201609,201610,201611,201612,201701,201702,201703,201704,201705,201706,201707,201708,201709,201710,201711,201712,201801,201802,201803,201804,201805,201806,201807,201808,201809,201810,201811,201812,201901,201902,201903,201904,201905,201906,201907,201908,201909,201910,201911,201912,202001,202002,202003,202004,202005,202006,202007,202008,202009,202010,202011,202012,202101,202102,202103,202104,202105,202106,202107,202108,202109,202110,202111,202112,202201,202202,202203,202204,202205,202206,202207,202208,202209,202210,202211,202212,202301,202302,202303,202304,202305,202306,202307,202308,202309,202310,202311,202312,202401,202402,202403,202404,202405,202406,202407,202408,202409,202410,202411,202412,202501,202502,202503,202504,202505,202506,202507,202508,202509,202510,202511,202512
fbi_code_desc,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Aggravated Assault,0.190405,0.108216,1.086270,1.415027,1.357494,1.571187,2.130075,1.990353,1.719128,1.078051,0.584914,-0.097257,0.436973,-0.450672,0.724636,1.135583,1.727347,1.858850,1.932820,1.530092,1.834193,1.143802,-0.047944,0.067121,-0.105476,-0.442453,0.420536,0.363003,1.176678,1.078051,1.990353,1.513654,1.308181,1.102707,0.486287,0.132873,-0.327388,-0.097257,0.790388,0.527382,1.579406,1.793098,1.719128,1.127364,1.176678,0.913672,0.001370,-0.516423,-0.952027,-0.467110,0.223281,0.913672,1.045175,1.431465,1.299962,0.864358,0.453411,0.453411,-0.302731,-1.017778,0.132873,-0.968465,-0.105476,0.469849,0.847920,0.576695,1.283524,0.930110,0.338346,0.124654,-0.491766,-0.483547,-1.009559,-1.297222,0.239719,0.124654,0.675323,0.650666,0.880796,0.683542,0.650666,0.280814,-0.606832,-0.771210,-0.656145,-1.206814,-0.253417,0.650666,0.428755,0.749293,1.127364,0.962985,0.502725,-0.031506,-1.025997,-1.445163,-0.976684,-1.034216,0.083559,0.256157,0.280814,0.313689,0.478068,0.617790,0.017808,-0.261636,-0.721897,-1.305441,-1.297222,-1.404068,-0.615050,-0.171228,-0.039725,-0.261636,-0.163009,-0.730116,-0.245198,-0.541080,-0.804086,-1.913643,-1.469820,-1.708169,-1.075311,-0.565737,-0.360263,-0.097257,0.165749,-0.499985,-0.352044,-0.779429,-1.256128,-1.182157,-2.061583,-1.856110,-0.483547,-0.993122,-0.623269,-0.327388,-0.310950,-0.442453,-0.869838,-0.935589,-1.223252,-1.387631,-1.716388,-1.897205,-1.552009,-1.289003,-0.828743,-0.976684,-1.091749,-0.894494,-1.305441,-1.338317,-1.773921,-1.823234,-1.971175,-2.447873,-1.716388,-1.486258,-0.746553,-0.713678,-0.795867,-0.8